<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Fake_news_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1_Step_1

In [4]:
"""
================================================================================
FAKE NEWS XAI — PHASE 1, STEP 1  (v4 — THE BULLETPROOF LOADER)
Fixes: Corrected Tariq60 dataset path, Bypassed HF deprecation errors.
================================================================================
"""
import subprocess, sys, os, io, zipfile, requests, warnings
warnings.filterwarnings("ignore")

for pkg in ["wordcloud", "kaleido", "datasets"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from datasets import load_dataset

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi":150,"font.family":"DejaVu Sans"})

print("="*70)
print(f"  Python: {sys.version.split()[0]}  | PyTorch: {torch.__version__}  | "
      f"CUDA: {'Tesla T4' if torch.cuda.is_available() else 'CPU'}")
print("="*70)

# ── Project dirs (idempotent) ─────────────────────────────────────────────────
PROJECT_ROOT = "/content/fake_news_xai"
for d in ["data/raw/liar","data/raw/fakenewsnet","data/raw/fakeddit",
          "data/processed/text","data/processed/images","data/processed/metadata",
          "models/base/text_bert","models/base/image_vit","models/base/metadata_xgb",
          "models/meta_learner","xai/shap","xai/lime","xai/gradcam","xai/attention",
          "results/metrics","results/visualizations","results/explanations",
          "src/preprocessing","src/models","src/xai","src/evaluation",
          "notebooks","logs","checkpoints"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print(f"✓ Project tree at {PROJECT_ROOT}\n")

# ── Label constants ───────────────────────────────────────────────────────────
LABEL_NAMES      = {0:"pants-fire",1:"false",2:"barely-true",
                   3:"half-true", 4:"mostly-true",5:"true"}
LABEL_STR_TO_INT = {v:k for k,v in LABEL_NAMES.items()}
BINARY_MAP       = {0:0,1:0,2:0,3:1,4:1,5:1}

LIAR_14_COLS = ["id","label","statement","subject","speaker","job_title",
                "state","party","barely_true_ct","false_ct","half_true_ct",
                "mostly_true_ct","pants_fire_ct","context"]

# =============================================================================
# STRATEGY 1 — LIAR-PLUS v2 TSVs (Corrected /dataset/ Path)
# =============================================================================
def strategy_liar_plus_github():
    # Fix: Added 'dataset/' to the base URL
    BASE   = "https://raw.githubusercontent.com/Tariq60/LIAR-PLUS/master/dataset/"
    FNAMES = {"train":"train2.tsv","validation":"val2.tsv","test":"test2.tsv"}
    dfs    = {}
    for split, fname in FNAMES.items():
        url  = BASE + fname
        resp = requests.get(url, timeout=30); resp.raise_for_status()
        df   = pd.read_csv(io.StringIO(resp.text), sep="\t",
                           header=None, on_bad_lines="skip")
        ncols = df.shape[1]

        cols = LIAR_14_COLS.copy()
        if ncols >= 15: cols.append("justification")
        if ncols >= 16: cols.append("extra")
        df.columns = cols[:ncols]

        # Robust string matching
        df["label"] = df["label"].astype(str).str.strip().str.lower().map(LABEL_STR_TO_INT)
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].astype(int)
        dfs[split]  = df
        print(f"    ✓  {split:12s}: {len(df):,} rows, {ncols} columns detected")
    return dfs

# =============================================================================
# STRATEGY 2 — Direct download from UCSB (original LIAR fallback)
# =============================================================================
def strategy_ucsb_direct():
    url  = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
    print(f"    Downloading from UCSB: {url}")
    r    = requests.get(url, timeout=60); r.raise_for_status()
    zf   = zipfile.ZipFile(io.BytesIO(r.content))
    FILE_MAP = {"train.tsv":"train","valid.tsv":"validation","test.tsv":"test"}
    dfs = {}
    for fname, split in FILE_MAP.items():
        with zf.open(fname) as f:
            df = pd.read_csv(f, sep="\t", header=None,
                             names=LIAR_14_COLS, on_bad_lines="skip")
        df["label"] = df["label"].astype(str).str.strip().str.lower().map(LABEL_STR_TO_INT)
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].astype(int)
        dfs[split] = df
        print(f"    ✓  {split:12s}: {len(df):,} rows (UCSB direct, no justifications)")
    return dfs

# =============================================================================
# RUN WATERFALL
# =============================================================================
STRATEGIES = [
    ("LIAR-PLUS GitHub (Direct Request)", strategy_liar_plus_github),
    ("UCSB Direct Zip Download", strategy_ucsb_direct),
]

print("Loading LIAR dataset — executing waterfall:\n")
dfs, DATASET_SOURCE = None, ""
for name, fn in STRATEGIES:
    print(f"  ▶  Trying: {name}")
    try:
        dfs = fn()
        DATASET_SOURCE = name
        print(f"\n  ✓  SUCCESS via: {name}\n")
        break
    except Exception as e:
        print(f"  ✗  Failed: {type(e).__name__}: {str(e)[:120]}\n")

if dfs is None:
    raise RuntimeError("All loading strategies failed. Check network connectivity.")

train_df, val_df, test_df = dfs["train"], dfs["validation"], dfs["test"]

# ─────────────────────────────────────────────────────────────────────────────
# FEATURE ENRICHMENT
# ─────────────────────────────────────────────────────────────────────────────
def enrich(df):
    df = df.copy()
    df = df[df["label"].between(0,5)].reset_index(drop=True)
    df["label_name"]   = df["label"].map(LABEL_NAMES)
    df["binary_label"] = df["label"].map(BINARY_MAP)
    df["binary_name"]  = df["binary_label"].map({0:"Fake",1:"Real"})
    df["stmt_chars"]   = df["statement"].astype(str).str.len()
    df["stmt_words"]   = df["statement"].astype(str).str.split().str.len()
    if "justification" in df.columns:
        df["just_words"] = df["justification"].astype(str).str.split().str.len()
    return df

train_df = enrich(train_df)
val_df   = enrich(val_df)
test_df  = enrich(test_df)
full_df  = pd.concat([train_df,val_df,test_df], ignore_index=True)

HAS_JUST = "justification" in full_df.columns

# ─────────────────────────────────────────────────────────────────────────────
# EDA — CONSOLE
# ─────────────────────────────────────────────────────────────────────────────
print("="*70)
print(f"  EDA  |  Source: {DATASET_SOURCE}")
print("="*70)
print(f"\n  Total: {len(full_df):,}  "
      f"(train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,})")

print("  6-Class Distribution:")
for k in range(6):
    n   = (full_df["label"]==k).sum()
    bar = "█"*int(n/len(full_df)*40)
    print(f"  {LABEL_NAMES[k]:>15s}  {n:>5,} ({100*n/len(full_df):5.1f}%)  {bar}")

print("\n  Binary Distribution:")
for name, n in full_df["binary_name"].value_counts().items():
    bar = "█"*int(n/len(full_df)*40)
    print(f"  {'FAKE' if name=='Fake' else 'REAL':>6s}  {n:>5,} ({100*n/len(full_df):5.1f}%)  {bar}")

print(f"\n  Statement length (words): "
      f"μ={full_df['stmt_words'].mean():.1f}  "
      f"σ={full_df['stmt_words'].std():.1f}  "
      f"max={full_df['stmt_words'].max()}")

if HAS_JUST:
    print(f"  Justification length (words): "
          f"μ={full_df['just_words'].mean():.1f}  "
          f"max={full_df['just_words'].max()}")

nulls = full_df.isnull().sum()
nonzero = nulls[nulls>0]
print(f"\n  Null check: {'CLEAN' if len(nonzero)==0 else nonzero.to_string()}")

# ─────────────────────────────────────────────────────────────────────────────
# SAVE PROCESSED CSVs
# ─────────────────────────────────────────────────────────────────────────────
PROC = os.path.join(PROJECT_ROOT,"data","processed","text")
for df,name in [(train_df,"train"),(val_df,"val"),(test_df,"test")]:
    path = os.path.join(PROC,f"liar_{name}.csv")
    df.to_csv(path, index=False)
    print(f"  ✓ Saved liar_{name}.csv  ({len(df):,} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# FakeNewsNet (Optional Extra)
# ─────────────────────────────────────────────────────────────────────────────
print("\n"+"="*70)
print("  FakeNewsNet  (rickstello/FakeNewsNet)")
print("="*70)
try:
    fnn    = load_dataset("rickstello/FakeNewsNet")
    fnn_df = fnn[list(fnn.keys())[0]].to_pandas()
    print(f"  ✓  {len(fnn_df):,} rows | cols: {list(fnn_df.columns)}")
    fnn_df.to_csv(os.path.join(PROJECT_ROOT,"data","raw","fakenewsnet","fakenewsnet.csv"), index=False)
except Exception as e:
    print(f"  ✗  Skipping FakeNewsNet for now: {str(e)[:120]}")

print(f"\n{'='*70}\n  PHASE 1, STEP 1 — COMPLETE\n{'='*70}")

  Python: 3.12.12  | PyTorch: 2.10.0+cu128  | CUDA: Tesla T4
✓ Project tree at /content/fake_news_xai

Loading LIAR dataset — executing waterfall:

  ▶  Trying: LIAR-PLUS GitHub (Direct Request)
  ✗  Failed: HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/Tariq60/LIAR-PLUS/master/dataset/train2.tsv

  ▶  Trying: UCSB Direct Zip Download
    ✓  train       : 10,240 rows (UCSB direct, no justifications)
    ✓  validation  : 1,284 rows (UCSB direct, no justifications)
    ✓  test        : 1,267 rows (UCSB direct, no justifications)

  ✓  SUCCESS via: UCSB Direct Zip Download

  EDA  |  Source: UCSB Direct Zip Download

  Total: 12,791  (train 10,240 | val 1,284 | test 1,267)
  6-Class Distribution:
       pants-fire  1,047 (  8.2%)  ███
            false  2,507 ( 19.6%)  ███████
      barely-true  2,103 ( 16.4%)  ██████
        half-true  2,627 ( 20.5%)  ████████
      mostly-true  2,454 ( 19.2%)  ███████
             true  2,053 ( 16.1%)  ██████

  Binar

README.md:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

FakeNewsNet.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/23196 [00:00<?, ? examples/s]

  ✓  23,196 rows | cols: ['title', 'news_url', 'source_domain', 'tweet_num', 'real']

  PHASE 1, STEP 1 — COMPLETE


In [5]:
"""
================================================================================
FAKE NEWS XAI — PHASE 1, STEP 2
BERT Tokenizer · Null Imputation · Metadata Features · PyTorch DataLoaders
================================================================================
Inputs  : /content/fake_news_xai/data/processed/text/liar_{train,val,test}.csv
          FakeNewsNet (in-memory from HuggingFace)
Outputs : Tokenized .pt tensors, metadata .npy arrays, DataLoader smoke test
================================================================================
"""
import os, warnings, pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from datasets import load_dataset
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = "/content/fake_news_xai"
PROC_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "text")
META_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "metadata")
VIS_DIR      = os.path.join(PROJECT_ROOT, "results", "visualizations")
os.makedirs(META_DIR, exist_ok=True)

print(f"Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print("="*70)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  LOAD SAVED CSVs
# ─────────────────────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROC_DIR, "liar_train.csv"))
val_df   = pd.read_csv(os.path.join(PROC_DIR, "liar_val.csv"))
test_df  = pd.read_csv(os.path.join(PROC_DIR, "liar_test.csv"))

print(f"Loaded CSVs — train:{len(train_df):,}  val:{len(val_df):,}  test:{len(test_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  NULL IMPUTATION  (deliberate strategy per column semantics)
# ─────────────────────────────────────────────────────────────────────────────
#
# job_title (3,568 missing): high missingness → sentinel "unknown_role"
# state     (2,751 missing): not recoverable  → sentinel "unknown_state"
# context   (131 missing)  : low missingness  → sentinel "unknown_context"
# subject, speaker, party  (2 each): fill with mode
# credit counts (2 each)   : these are history totals; 0 is semantically correct
#
CAT_SENTINELS = {
    "job_title" : "unknown_role",
    "state"     : "unknown_state",
    "context"   : "unknown_context",
    "subject"   : "unknown_subject",
    "speaker"   : "unknown_speaker",
    "party"     : "unknown_party",
}
NUM_COLS = ["barely_true_ct","false_ct","half_true_ct","mostly_true_ct","pants_fire_ct"]

def impute(df, fit_modes=None):
    df = df.copy()
    modes = fit_modes if fit_modes else {}
    for col, sentinel in CAT_SENTINELS.items():
        if col in df.columns:
            if col not in modes:
                modes[col] = df[col].mode()[0] if df[col].notna().any() else sentinel
            df[col] = df[col].fillna(sentinel)   # use sentinel, not mode, for
    for col in NUM_COLS:                          # interpretability in XGBoost
        if col in df.columns:
            df[col] = df[col].fillna(0.0)
    return df, modes

train_df, MODES = impute(train_df)
val_df,   _     = impute(val_df,   MODES)
test_df,  _     = impute(test_df,  MODES)

# Verify
remaining_nulls = train_df.isnull().sum().sum()
print(f"\nPost-imputation null count (train): {remaining_nulls}  "
      f"{'✓ CLEAN' if remaining_nulls==0 else '✗ CHECK'}")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  TOKENIZER SETUP
# ─────────────────────────────────────────────────────────────────────────────
#
# MAX_LENGTH choice rationale:
#   LIAR mean=18 words, std=10 → 99th percentile ≈ 18+3×10 = 48 words ≈ 60 tokens.
#   We use MAX_LENGTH=128 to accommodate long-tail (max 467 words) and
#   future FakeNewsNet title concatenation without truncating meaning.
#   BERT handles [CLS] statement [SEP] within 128 comfortably.
#
MAX_LENGTH = 128
MODEL_NAME = "bert-base-uncased"

print(f"\nLoading tokenizer: {MODEL_NAME}  (MAX_LENGTH={MAX_LENGTH})")
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f"  Vocab size   : {tokenizer.vocab_size:,}")
print(f"  Special tokens: {tokenizer.all_special_tokens}")

# Verify token length coverage
sample_lens = train_df["statement"].astype(str).apply(
    lambda x: len(tokenizer.encode(x, truncation=False))
)
trunc_rate = (sample_lens > MAX_LENGTH).mean() * 100
print(f"\n  Token length stats (train statements):")
print(f"    μ={sample_lens.mean():.1f}  σ={sample_lens.std():.1f}  "
      f"max={sample_lens.max()}  p95={np.percentile(sample_lens,95):.0f}  "
      f"p99={np.percentile(sample_lens,99):.0f}")
print(f"    Truncation rate at MAX_LENGTH={MAX_LENGTH}: {trunc_rate:.2f}%  "
      f"({'✓ acceptable' if trunc_rate < 1 else '⚠ consider increasing'})")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  TOKENIZATION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def tokenize_batch(texts, max_length=MAX_LENGTH):
    """
    Returns dict of numpy arrays: input_ids, attention_mask, token_type_ids.
    Using numpy for storage efficiency; converted to tensors inside Dataset.__getitem__.
    """
    enc = tokenizer(
        list(texts),
        padding     = "max_length",
        truncation  = True,
        max_length  = max_length,
        return_tensors = "np",
    )
    return {
        "input_ids"      : enc["input_ids"].astype(np.int32),
        "attention_mask" : enc["attention_mask"].astype(np.int8),
        "token_type_ids" : enc.get("token_type_ids",
                            np.zeros_like(enc["input_ids"])).astype(np.int8),
    }

print("\nTokenizing splits (batch=512)...")

def tokenize_split(df, batch_size=512):
    texts = df["statement"].astype(str).tolist()
    all_ids, all_mask, all_tt = [], [], []
    for i in range(0, len(texts), batch_size):
        batch = tokenize_batch(texts[i:i+batch_size])
        all_ids.append(batch["input_ids"])
        all_mask.append(batch["attention_mask"])
        all_tt.append(batch["token_type_ids"])
        if (i // batch_size) % 5 == 0:
            print(f"    {i+batch_size:>6}/{len(texts)}", end="\r")
    return {
        "input_ids"      : np.vstack(all_ids),
        "attention_mask" : np.vstack(all_mask),
        "token_type_ids" : np.vstack(all_tt),
    }

tok_train = tokenize_split(train_df)
tok_val   = tokenize_split(val_df)
tok_test  = tokenize_split(test_df)

for split_name, tok in [("train",tok_train),("val",tok_val),("test",tok_test)]:
    print(f"  ✓ {split_name:6s}: input_ids {tok['input_ids'].shape}  "
          f"dtype {tok['input_ids'].dtype}  "
          f"mem {tok['input_ids'].nbytes/1e6:.1f} MB")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  METADATA FEATURE ENGINEERING  (for XGBoost base learner)
# ─────────────────────────────────────────────────────────────────────────────
#
# Features:
#   A) Credit history (5 numeric counts) → normalized
#   B) Derived credibility ratio: false_rate = (pants_fire + false) / (total + 1)
#   C) Categorical: party (21 unique), state (50+), job_title (3k+ → top-K + other)
#   D) Statement-level: token_count (proxy for claim specificity)
#
# job_title has 3k+ unique values → top-100 + "other" to avoid cardinality explosion.
# This is consistent with Theory of Content Consistency (ToCC): source metadata
# (who, from where, which party) encodes structural authenticity signals.
#

print("\n" + "="*70)
print("  METADATA FEATURE ENGINEERING")
print("="*70)

# ── 5a. Cardinality report ────────────────────────────────────────────────────
META_CAT_COLS = ["party","state","job_title"]
print("\n  Categorical cardinality (train):")
for col in META_CAT_COLS:
    print(f"    {col:15s}: {train_df[col].nunique():,} unique values")

# ── 5b. Reduce job_title cardinality → top 100 + 'other' ─────────────────────
TOP_K_JOB = 100
top_jobs = train_df["job_title"].value_counts().head(TOP_K_JOB).index.tolist()
def cap_job_title(df):
    df = df.copy()
    df["job_title_capped"] = df["job_title"].apply(
        lambda x: x if x in top_jobs else "other_role"
    )
    return df

train_df = cap_job_title(train_df)
val_df   = cap_job_title(val_df)
test_df  = cap_job_title(test_df)
print(f"\n  job_title capped to top-{TOP_K_JOB} + 'other_role' "
      f"→ {train_df['job_title_capped'].nunique()} categories")

# ── 5c. Label Encoding (fit on train only) ────────────────────────────────────
LE_COLS    = ["party", "state", "job_title_capped"]
ENCODERS   = {}
for col in LE_COLS:
    le = LabelEncoder()
    train_df[f"{col}_enc"] = le.fit_transform(train_df[col].astype(str))
    val_df  [f"{col}_enc"] = le.transform(
        val_df[col if col != "job_title_capped" else col].astype(str).apply(
            lambda x: x if x in le.classes_ else le.classes_[0]))
    test_df [f"{col}_enc"] = le.transform(
        test_df[col if col != "job_title_capped" else col].astype(str).apply(
            lambda x: x if x in le.classes_ else le.classes_[0]))
    ENCODERS[col] = le

# ── 5d. Derived credibility features ─────────────────────────────────────────
def add_credibility_features(df):
    df = df.copy()
    df["total_claims"]    = df[NUM_COLS].sum(axis=1)
    df["false_rate"]      = (df["pants_fire_ct"] + df["false_ct"]) / (df["total_claims"] + 1)
    df["credibility_score"] = (df["mostly_true_ct"] + df["half_true_ct"] * 0.5) / (df["total_claims"] + 1)
    df["extreme_rate"]    = (df["pants_fire_ct"]) / (df["total_claims"] + 1)
    return df

train_df = add_credibility_features(train_df)
val_df   = add_credibility_features(val_df)
test_df  = add_credibility_features(test_df)

DERIVED_COLS = ["total_claims","false_rate","credibility_score","extreme_rate"]
print(f"\n  Derived credibility features added: {DERIVED_COLS}")
print(f"  False rate stats (train): "
      f"μ={train_df['false_rate'].mean():.3f}  "
      f"σ={train_df['false_rate'].std():.3f}  "
      f"max={train_df['false_rate'].max():.3f}")

# ── 5e. Assemble metadata matrix and scale ────────────────────────────────────
META_FEATURES = (
    NUM_COLS
    + DERIVED_COLS
    + ["stmt_words"]
    + [f"{c}_enc" for c in LE_COLS]
)

def build_meta_matrix(df):
    return df[META_FEATURES].values.astype(np.float32)

X_meta_train = build_meta_matrix(train_df)
X_meta_val   = build_meta_matrix(val_df)
X_meta_test  = build_meta_matrix(test_df)

scaler  = StandardScaler()
X_meta_train_scaled = scaler.fit_transform(X_meta_train)
X_meta_val_scaled   = scaler.transform(X_meta_val)
X_meta_test_scaled  = scaler.transform(X_meta_test)

print(f"\n  Metadata matrix shape: train={X_meta_train_scaled.shape}  "
      f"val={X_meta_val_scaled.shape}  test={X_meta_test_scaled.shape}")
print(f"  Features ({len(META_FEATURES)}): {META_FEATURES}")

# ── 5f. Class weights for imbalance handling ──────────────────────────────────
y_train = train_df["binary_label"].values
cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
print(f"\n  Class weights (Fake=0, Real=1): "
      f"Fake={CLASS_WEIGHTS[0]:.4f}  Real={CLASS_WEIGHTS[1]:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  FAKENEWSNET PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FAKENEWSNET PREPROCESSING")
print("="*70)

fnn = load_dataset("rickstello/FakeNewsNet")
fnn_df = fnn[list(fnn.keys())[0]].to_pandas()

# Label check
print(f"\n  'real' column distribution:\n"
      f"  {fnn_df['real'].value_counts().to_dict()}")
print(f"  (1 = Real news, 0 = Fake news  — same convention as LIAR binary_label)")

# Null check
print(f"\n  Null values:\n{fnn_df.isnull().sum()[fnn_df.isnull().sum()>0].to_string()}")

# Impute
fnn_df["title"]  = fnn_df["title"].fillna("").astype(str)
fnn_df["source_domain"] = fnn_df["source_domain"].fillna("unknown_source")
fnn_df["tweet_num"]     = fnn_df["tweet_num"].fillna(0).astype(int)
fnn_df["binary_label"]  = fnn_df["real"].astype(int)
fnn_df["binary_name"]   = fnn_df["binary_label"].map({0:"Fake",1:"Real"})
fnn_df["title_words"]   = fnn_df["title"].str.split().str.len()

print(f"\n  FakeNewsNet post-imputation: {len(fnn_df):,} rows")
print(f"  Title length: μ={fnn_df['title_words'].mean():.1f}  "
      f"max={fnn_df['title_words'].max()}")
print(f"  Binary: Fake={int((fnn_df['binary_label']==0).sum()):,}  "
      f"Real={int((fnn_df['binary_label']==1).sum()):,}")
print(f"  Source domains: {fnn_df['source_domain'].nunique():,} unique")
print(f"  tweet_num range: [{fnn_df['tweet_num'].min()}, {fnn_df['tweet_num'].max()}]")

# Train/val/test split for FakeNewsNet (80/10/10 stratified)
from sklearn.model_selection import train_test_split

fnn_train, fnn_temp = train_test_split(
    fnn_df, test_size=0.20, stratify=fnn_df["binary_label"], random_state=42)
fnn_val, fnn_test = train_test_split(
    fnn_temp, test_size=0.50, stratify=fnn_temp["binary_label"], random_state=42)

print(f"\n  FakeNewsNet splits: train={len(fnn_train):,}  "
      f"val={len(fnn_val):,}  test={len(fnn_test):,}")

# Tokenize FNN titles
print("\n  Tokenizing FakeNewsNet titles...")
tok_fnn_train = tokenize_split(fnn_train.rename(columns={"title":"statement"}))
tok_fnn_val   = tokenize_split(fnn_val.rename(  columns={"title":"statement"}))
tok_fnn_test  = tokenize_split(fnn_test.rename( columns={"title":"statement"}))
print(f"  ✓ FNN tokenization complete: {tok_fnn_train['input_ids'].shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 7.  PYTORCH DATASET CLASSES
# ─────────────────────────────────────────────────────────────────────────────
class LIARDataset(Dataset):
    """
    Returns:
      input_ids      (max_length,)  int32 → BERT text branch
      attention_mask (max_length,)  int8
      token_type_ids (max_length,)  int8
      metadata       (n_meta_feats,) float32 → XGBoost / meta-learner branch
      label          scalar int64  (binary)
    """
    def __init__(self, tok_dict, meta_matrix, labels):
        assert len(labels) == tok_dict["input_ids"].shape[0] == meta_matrix.shape[0]
        self.input_ids      = tok_dict["input_ids"]
        self.attention_mask = tok_dict["attention_mask"]
        self.token_type_ids = tok_dict["token_type_ids"]
        self.metadata       = meta_matrix.astype(np.float32)
        self.labels         = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : torch.tensor(self.input_ids[idx],      dtype=torch.long),
            "attention_mask" : torch.tensor(self.attention_mask[idx],  dtype=torch.long),
            "token_type_ids" : torch.tensor(self.token_type_ids[idx],  dtype=torch.long),
            "metadata"       : torch.tensor(self.metadata[idx],        dtype=torch.float),
            "label"          : torch.tensor(self.labels[idx],          dtype=torch.long),
        }


class FakeNewsNetDataset(Dataset):
    """Title-only FakeNewsNet dataset (image branch added in Phase 2)."""
    def __init__(self, tok_dict, df):
        self.input_ids      = tok_dict["input_ids"]
        self.attention_mask = tok_dict["attention_mask"]
        self.token_type_ids = tok_dict["token_type_ids"]
        self.tweet_num      = df["tweet_num"].values.astype(np.float32)
        self.labels         = df["binary_label"].values.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : torch.tensor(self.input_ids[idx],     dtype=torch.long),
            "attention_mask" : torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "token_type_ids" : torch.tensor(self.token_type_ids[idx], dtype=torch.long),
            "tweet_num"      : torch.tensor([self.tweet_num[idx]],    dtype=torch.float),
            "label"          : torch.tensor(self.labels[idx],         dtype=torch.long),
        }

# Instantiate datasets
liar_train_ds = LIARDataset(tok_train, X_meta_train_scaled, y_train)
liar_val_ds   = LIARDataset(tok_val,   X_meta_val_scaled,
                             val_df["binary_label"].values)
liar_test_ds  = LIARDataset(tok_test,  X_meta_test_scaled,
                             test_df["binary_label"].values)

fnn_train_ds  = FakeNewsNetDataset(tok_fnn_train, fnn_train.reset_index(drop=True))
fnn_val_ds    = FakeNewsNetDataset(tok_fnn_val,   fnn_val.reset_index(drop=True))
fnn_test_ds   = FakeNewsNetDataset(tok_fnn_test,  fnn_test.reset_index(drop=True))

# DataLoaders
BATCH_SIZE = 32   # conservative for T4 (15 GB VRAM); increase to 64 if memory allows

liar_train_loader = DataLoader(liar_train_ds, batch_size=BATCH_SIZE,
                                shuffle=True,  num_workers=2, pin_memory=True)
liar_val_loader   = DataLoader(liar_val_ds,   batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)
liar_test_loader  = DataLoader(liar_test_ds,  batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)
fnn_train_loader  = DataLoader(fnn_train_ds,  batch_size=BATCH_SIZE,
                                shuffle=True,  num_workers=2, pin_memory=True)

print("\n" + "="*70)
print("  PYTORCH DATASET / DATALOADER SMOKE TEST")
print("="*70)

# Smoke test — pull one batch and verify shapes
batch = next(iter(liar_train_loader))
print(f"\n  LIAR train batch (size={BATCH_SIZE}):")
for k, v in batch.items():
    print(f"    {k:20s}: shape={tuple(v.shape)}  dtype={v.dtype}")

batch_fnn = next(iter(fnn_train_loader))
print(f"\n  FakeNewsNet train batch (size={BATCH_SIZE}):")
for k, v in batch_fnn.items():
    print(f"    {k:20s}: shape={tuple(v.shape)}  dtype={v.dtype}")

# ─────────────────────────────────────────────────────────────────────────────
# 8.  SAVE ALL PREPROCESSING ARTIFACTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  SAVING PREPROCESSING ARTIFACTS")
print("="*70)

# Tokenized arrays
for split, tok in [("train",tok_train),("val",tok_val),("test",tok_test)]:
    for key, arr in tok.items():
        path = os.path.join(PROC_DIR, f"liar_{split}_{key}.npy")
        np.save(path, arr)
print(f"  ✓ LIAR tokenized arrays  → {PROC_DIR}/liar_{{split}}_{{key}}.npy")

# Metadata matrices
for split, mat in [("train",X_meta_train_scaled),
                   ("val",  X_meta_val_scaled),
                   ("test", X_meta_test_scaled)]:
    np.save(os.path.join(META_DIR, f"liar_{split}_meta.npy"), mat)
print(f"  ✓ Metadata matrices      → {META_DIR}/")

# Encoders + scaler + class weights
artifacts = {
    "label_encoders"  : ENCODERS,
    "scaler"          : scaler,
    "top_jobs"        : top_jobs,
    "meta_features"   : META_FEATURES,
    "max_length"      : MAX_LENGTH,
    "class_weights"   : cw,
    "binary_map"      : {0:0,1:0,2:0,3:1,4:1,5:1},
}
with open(os.path.join(META_DIR, "preprocessing_artifacts.pkl"), "wb") as f:
    pickle.dump(artifacts, f)
print(f"  ✓ Preprocessing artifacts → {META_DIR}/preprocessing_artifacts.pkl")

# Processed FNN splits
fnn_train.to_csv(os.path.join(PROC_DIR,"fnn_train.csv"),index=False)
fnn_val.to_csv(  os.path.join(PROC_DIR,"fnn_val.csv"),  index=False)
fnn_test.to_csv( os.path.join(PROC_DIR,"fnn_test.csv"), index=False)
print(f"  ✓ FakeNewsNet splits     → {PROC_DIR}/fnn_{{train,val,test}}.csv")

# ─────────────────────────────────────────────────────────────────────────────
# 9.  METADATA FEATURE CORRELATION HEATMAP
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Metadata Feature Analysis", fontsize=14, fontweight="bold")

# Correlation matrix of numeric metadata features vs binary label
numeric_check_cols = NUM_COLS + DERIVED_COLS + ["stmt_words","binary_label"]
corr = train_df[numeric_check_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=axes[0],
            annot_kws={"size":7})
axes[0].set_title("Feature Correlation Matrix", fontsize=11)
axes[0].tick_params(axis="x", rotation=45)

# False rate distribution by class
for label, color, name in [(0,"#d62728","Fake"),(1,"#2ca02c","Real")]:
    subset = train_df[train_df["binary_label"]==label]["false_rate"]
    axes[1].hist(subset, bins=40, color=color, alpha=0.65,
                 label=f"{name} (n={len(subset):,})", edgecolor="white")
axes[1].set_title("Speaker False Rate Distribution by Class")
axes[1].set_xlabel("false_rate  =  (pants_fire + false) / (total_claims + 1)")
axes[1].set_ylabel("Count"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR,"06_metadata_features.png"), bbox_inches="tight")
plt.close(); print(f"\n  ✓ Fig 6 saved: 06_metadata_features.png")

# ─────────────────────────────────────────────────────────────────────────────
# 10.  FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
{"="*70}
  PHASE 1, STEP 2 — COMPLETE
{"="*70}
  LIAR Datasets
  ├── LIARDataset     : train={len(liar_train_ds):,}  val={len(liar_val_ds):,}  test={len(liar_test_ds):,}
  ├── BERT input shape: (batch, {MAX_LENGTH})
  ├── Metadata shape  : (batch, {len(META_FEATURES)})
  └── Class weights   : Fake={cw[0]:.4f}  Real={cw[1]:.4f}

  FakeNewsNet Datasets
  ├── FakeNewsNetDataset: train={len(fnn_train_ds):,}  val={len(fnn_val_ds):,}  test={len(fnn_test_ds):,}
  └── BERT input shape  : (batch, {MAX_LENGTH})

  DataLoaders: batch_size={BATCH_SIZE}  num_workers=2  pin_memory=True (GPU)

  ──────────────────────────────────────────────────────────────────────
  NEXT → Phase 2, Step 1: BERT Text Base Learner (fine-tuning + training)
  ──────────────────────────────────────────────────────────────────────
""")


Device: cuda  |  GPU: Tesla T4
Loaded CSVs — train:10,240  val:1,284  test:1,267

Post-imputation null count (train): 0  ✓ CLEAN

Loading tokenizer: bert-base-uncased  (MAX_LENGTH=128)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Vocab size   : 30,522
  Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


Token indices sequence length is longer than the specified maximum sequence length for this model (714 > 512). Running this sequence through the model will result in indexing errors



  Token length stats (train statements):
    μ=24.3  σ=12.9  max=714  p95=42  p99=53
    Truncation rate at MAX_LENGTH=128: 0.03%  (✓ acceptable)

Tokenizing splits (batch=512)...
  ✓ train : input_ids (10240, 128)  dtype int32  mem 5.2 MB
  ✓ val   : input_ids (1284, 128)  dtype int32  mem 0.7 MB
  ✓ test  : input_ids (1267, 128)  dtype int32  mem 0.6 MB

  METADATA FEATURE ENGINEERING

  Categorical cardinality (train):
    party          : 24 unique values
    state          : 84 unique values
    job_title      : 1,184 unique values

  job_title capped to top-100 + 'other_role' → 101 categories

  Derived credibility features added: ['total_claims', 'false_rate', 'credibility_score', 'extreme_rate']
  False rate stats (train): μ=0.255  σ=0.204  max=0.857

  Metadata matrix shape: train=(10240, 13)  val=(1284, 13)  test=(1267, 13)
  Features (13): ['barely_true_ct', 'false_ct', 'half_true_ct', 'mostly_true_ct', 'pants_fire_ct', 'total_claims', 'false_rate', 'credibility_score', 'ex

In [9]:
"""
================================================================================
FAKE NEWS XAI — PHASE 2, STEP 1  (v3 — MULTI-SAMPLE DROPOUT + FOCAL LOSS)
================================================================================
"""
import os, gc, pickle, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import BertModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = "/content/fake_news_xai"
PROC_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "text")
META_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "metadata")
MODEL_DIR    = os.path.join(PROJECT_ROOT, "models", "base", "text_bert")
VIS_DIR      = os.path.join(PROJECT_ROOT, "results", "visualizations")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results", "metrics")
STACK_DIR    = os.path.join(PROJECT_ROOT, "results", "stacking")
for d in [STACK_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)
print(f"Device: {DEVICE}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ─────────────────────────────────────────────────────────────────────────────
# 1. RELOAD ARTIFACTS
# ─────────────────────────────────────────────────────────────────────────────
with open(os.path.join(META_DIR, "preprocessing_artifacts.pkl"), "rb") as f:
    artifacts = pickle.load(f)
CLASS_WEIGHTS = torch.tensor(artifacts["class_weights"], dtype=torch.float32).to(DEVICE)

def load_tok(split):
    base = os.path.join(PROC_DIR, f"liar_{split}")
    return {k: np.load(f"{base}_{k}.npy") for k in ["input_ids","attention_mask","token_type_ids"]}

tok_train = load_tok("train")
tok_val   = load_tok("val")
tok_test  = load_tok("test")

train_df = pd.read_csv(os.path.join(PROC_DIR, "liar_train.csv"))
val_df   = pd.read_csv(os.path.join(PROC_DIR, "liar_val.csv"))
test_df  = pd.read_csv(os.path.join(PROC_DIR, "liar_test.csv"))

y_train  = train_df["binary_label"].values.astype(np.int64)
y_val    = val_df  ["binary_label"].values.astype(np.int64)
y_test   = test_df ["binary_label"].values.astype(np.int64)

# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET + DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
class LIARTextDataset(Dataset):
    def __init__(self, tok, labels):
        self.ids  = tok["input_ids"]
        self.mask = tok["attention_mask"]
        self.tt   = tok["token_type_ids"]
        self.y    = labels
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {
            "input_ids"      : torch.tensor(self.ids[i],  dtype=torch.long),
            "attention_mask" : torch.tensor(self.mask[i], dtype=torch.long),
            "token_type_ids" : torch.tensor(self.tt[i],   dtype=torch.long),
            "label"          : torch.tensor(self.y[i],    dtype=torch.long),
        }

BATCH_SIZE = 64
train_loader = DataLoader(LIARTextDataset(tok_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(LIARTextDataset(tok_val, y_val), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(LIARTextDataset(tok_test, y_test), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
train_loader_noshuffle = DataLoader(LIARTextDataset(tok_train, y_train), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. ADVANCED ARCHITECTURE (MULTI-SAMPLE DROPOUT)
# ─────────────────────────────────────────────────────────────────────────────
class BertFakeNewsClassifierV3(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        H = self.bert.config.hidden_size

        # V3 Trick: Multi-Sample Dropout
        # 5 parallel dropouts to average out variance and regularize heavily
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in np.linspace(0.1, 0.5, 5)])
        self.classifier = nn.Linear(H, num_labels)

    def mean_pool(self, last_hidden, attention_mask):
        mask_exp = attention_mask.unsqueeze(-1).float()
        summed   = (last_hidden * mask_exp).sum(dim=1)
        lengths  = mask_exp.sum(dim=1).clamp(min=1e-9)
        return summed / lengths

    def forward(self, input_ids, attention_mask, token_type_ids=None, return_cls=False):
        out  = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls   = out.last_hidden_state[:, 0, :]
        mp    = self.mean_pool(out.last_hidden_state, attention_mask)
        fused = 0.5 * cls + 0.5 * mp

        # Apply multi-sample dropout
        logits = torch.mean(torch.stack([self.classifier(d(fused)) for d in self.dropouts], dim=0), dim=0)

        if return_cls: return logits, fused
        return logits

def freeze_bert_layers(model, freeze_up_to=4):
    for p in model.bert.embeddings.parameters(): p.requires_grad = False
    for i in range(freeze_up_to):
        for p in model.bert.encoder.layer[i].parameters(): p.requires_grad = False
    return model

def get_llrd_optimizer(model, base_lr=1e-5, head_lr=5e-5, decay=0.9, weight_decay=0.01, freeze_up_to=4):
    param_groups = []
    head_params  = list(model.classifier.named_parameters())
    param_groups += [{"params": [p for n,p in head_params if "bias" not in n], "lr": head_lr, "weight_decay": weight_decay},
                     {"params": [p for n,p in head_params if "bias" in n], "lr": head_lr, "weight_decay": 0.0}]

    pooler_params = list(model.bert.pooler.named_parameters())
    lr_pooler = base_lr * (decay ** 0)
    param_groups += [{"params": [p for n,p in pooler_params if p.requires_grad and "bias" not in n], "lr": lr_pooler, "weight_decay": weight_decay},
                     {"params": [p for n,p in pooler_params if p.requires_grad and "bias" in n], "lr": lr_pooler, "weight_decay": 0.0}]

    num_layers = len(model.bert.encoder.layer)
    for layer_idx in range(num_layers - 1, freeze_up_to - 1, -1):
        depth = num_layers - 1 - layer_idx
        lr_k  = base_lr * (decay ** depth)
        layer = model.bert.encoder.layer[layer_idx]
        lparams = list(layer.named_parameters())
        param_groups += [{"params": [p for n,p in lparams if p.requires_grad and "bias" not in n], "lr": lr_k, "weight_decay": weight_decay},
                         {"params": [p for n,p in lparams if p.requires_grad and "bias" in n], "lr": lr_k, "weight_decay": 0.0}]
    return torch.optim.AdamW(param_groups)

# ─────────────────────────────────────────────────────────────────────────────
# 4. CUSTOM FOCAL LOSS (V3 Trick)
# ─────────────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha # expects a tensor of class weights
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        return focal_loss.sum()

# ─────────────────────────────────────────────────────────────────────────────
# 5. TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=4, min_delta=5e-4):
        self.patience, self.min_delta = patience, min_delta
        self.best, self.counter, self.should_stop = -np.inf, 0, False
    def step(self, score, model, ckpt_path):
        if score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            torch.save(model.state_dict(), ckpt_path)
            return True
        self.counter += 1
        if self.counter >= self.patience: self.should_stop = True
        return False

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, preds, probs_list, labels = 0.0, [], [], []
    for b in loader:
        ids, mask, tt, y = b["input_ids"].to(device), b["attention_mask"].to(device), b["token_type_ids"].to(device), b["label"].to(device)
        with autocast():
            logits = model(ids, mask, tt)
            loss   = criterion(logits, y)
        p = F.softmax(logits.float(), dim=-1)
        loss_sum += loss.item() * y.size(0)
        preds.extend(p.argmax(1).cpu().numpy()); probs_list.extend(p.cpu().numpy()); labels.extend(y.cpu().numpy())
    y_true, y_pred, y_prob = np.array(labels), np.array(preds), np.array(probs_list)
    return {
        "loss"        : loss_sum / len(y_true),
        "accuracy"    : accuracy_score(y_true, y_pred),
        "f1_macro"    : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted" : f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc"     : roc_auc_score(y_true, y_prob[:, 1]),
    }, y_prob, y_true

# ─────────────────────────────────────────────────────────────────────────────
# 6. SETUP & EXECUTION
# ─────────────────────────────────────────────────────────────────────────────
FREEZE_N, LR_BERT, LR_HEAD, LLRD_DECAY, WEIGHT_DECAY = 4, 1e-5, 5e-5, 0.9, 0.01
NUM_EPOCHS, CHECKPOINT = 15, os.path.join(MODEL_DIR, "bert_v3_best.pt")

print("\n" + "="*70 + "\n  MODEL SETUP  (V3 — Focal Loss + Multi-Dropout)\n" + "="*70)
model = freeze_bert_layers(BertFakeNewsClassifierV3().to(DEVICE), freeze_up_to=FREEZE_N)
optimizer = get_llrd_optimizer(model, base_lr=LR_BERT, head_lr=LR_HEAD, decay=LLRD_DECAY, weight_decay=WEIGHT_DECAY, freeze_up_to=FREEZE_N)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

# Applying V3 Focal Loss
criterion = FocalLoss(alpha=CLASS_WEIGHTS, gamma=2.0)
amp_scaler, stopper = GradScaler(), EarlyStopping(patience=4)

history = {"train_loss":[], "val_loss":[], "val_acc":[], "val_f1":[], "val_auc":[], "gap":[]}
print("\n" + "="*70 + "\n  TRAINING\n" + "="*70)
print(f"  {'Ep':>3}  {'Tr-Loss':>8}  {'V-Loss':>7}  {'V-Acc':>7}  {'V-F1':>6}  {'V-AUC':>7}  {'Gap':>6}  {'★':>2}  {'s':>4}")
print("  " + "-"*70)

for epoch in range(1, NUM_EPOCHS + 1):
    t0, ep_loss, n_seen = time.time(), 0.0, 0
    model.train()

    for batch in train_loader:
        ids, mask, tt, y = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["token_type_ids"].to(DEVICE), batch["label"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(ids, mask, tt)
            loss   = criterion(logits, y)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        scheduler.step()

        ep_loss += loss.item() * y.size(0)
        n_seen  += y.size(0)

    tr_loss = ep_loss / n_seen
    val_met, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    gap = val_met["loss"] - tr_loss

    for k, v in zip(["train_loss","val_loss","val_acc","val_f1","val_auc","gap"], [tr_loss, val_met["loss"], val_met["accuracy"], val_met["f1_macro"], val_met["roc_auc"], gap]):
        history[k].append(v)

    flag = "✓" if stopper.step(val_met["f1_macro"], model, CHECKPOINT) else ""
    print(f"  {epoch:>3}  {tr_loss:>8.4f}  {val_met['loss']:>7.4f}  {val_met['accuracy']:>7.4f}  {val_met['f1_macro']:>6.4f}  {val_met['roc_auc']:>7.4f}  {gap:>+6.3f}  {flag:>2}  {time.time() - t0:>3.0f}s")

    if stopper.should_stop:
        print(f"\n  ► Early stopping at epoch {epoch}  (best={stopper.best:.4f})")
        break

# ─────────────────────────────────────────────────────────────────────────────
# 7. TEST EVALUATION & SAVING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70 + "\n  TEST SET EVALUATION\n" + "="*70)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval() # Ensure model is in eval mode for final testing
test_met, test_probs, test_labels_arr = evaluate(model, test_loader, criterion, DEVICE)
print(f"\n  Accuracy : {test_met['accuracy']:.4f}\n  F1       : {test_met['f1_macro']:.4f}\n  ROC-AUC  : {test_met['roc_auc']:.4f}")

@torch.no_grad()
def extract_probs(m, loader):
    m.eval(); out = []
    for b in loader:
        with autocast(): out.append(F.softmax(m(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), b["token_type_ids"].to(DEVICE)).float(), dim=-1).cpu().numpy())
    return np.vstack(out)

for name, arr in [("train", extract_probs(model, train_loader_noshuffle)), ("val", extract_probs(model, val_loader)), ("test", test_probs)]:
    np.save(os.path.join(STACK_DIR, f"bert_proba_{name}.npy"), arr)

summary = {"model":"bert-base-uncased v3", "best_val_f1":float(stopper.best), "test_accuracy":float(test_met["accuracy"]), "test_f1_macro":float(test_met["f1_macro"])}
with open(os.path.join(RESULTS_DIR, "bert_v3_results.pkl"), "wb") as f: pickle.dump(summary, f)
print("\n  ✓ Saved probabilities for Phase 2, Step 2 Metadata XGBoost Stacking.")

Device: cuda  (Tesla T4)

  MODEL SETUP  (V3 — Focal Loss + Multi-Dropout)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING
   Ep   Tr-Loss   V-Loss    V-Acc    V-F1    V-AUC     Gap   ★     s
  ----------------------------------------------------------------------
    1    0.1813   0.1715   0.5483  0.5254   0.6174  -0.010   ✓   51s
    2    0.1693   0.1665   0.5888  0.5761   0.6692  -0.003   ✓   48s
    3    0.1648   0.1702   0.6394  0.6392   0.6807  +0.005   ✓   50s
    4    0.1591   0.1656   0.6051  0.5968   0.6820  +0.007       46s
    5    0.1541   0.1656   0.6363  0.6349   0.6874  +0.011       45s
    6    0.1477   0.1678   0.6184  0.6151   0.6812  +0.020       45s
    7    0.1413   0.1801   0.6293  0.6282   0.6723  +0.039       45s

  ► Early stopping at epoch 7  (best=0.6392)

  TEST SET EVALUATION

  Accuracy : 0.6314
  F1       : 0.6274
  ROC-AUC  : 0.6787

  ✓ Saved probabilities for Phase 2, Step 2 Metadata XGBoost Stacking.


In [12]:
"""
================================================================================
FAKE NEWS XAI — PHASE 2, STEP 2
XGBoost Metadata Base Learner (NumPy 2.0 Compatible)
================================================================================
"""
import subprocess, sys, os, gc, pickle, warnings, time

# --- DEPENDENCY AUTO-INSTALLER ---
for pkg in ["xgboost", "shap", "optuna"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# ---------------------------------

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import optuna
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import calibration_curve
import matplotlib

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = "/content/fake_news_xai"
META_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "metadata")
PROC_DIR     = os.path.join(PROJECT_ROOT, "data", "processed", "text")
MODEL_DIR    = os.path.join(PROJECT_ROOT, "models", "base", "metadata_xgb")
VIS_DIR      = os.path.join(PROJECT_ROOT, "results", "visualizations")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results", "metrics")
STACK_DIR    = os.path.join(PROJECT_ROOT, "results", "stacking")
XAI_DIR      = os.path.join(PROJECT_ROOT, "xai", "shap")
for d in [MODEL_DIR, XAI_DIR]: os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  LOAD METADATA MATRICES + LABELS
# ─────────────────────────────────────────────────────────────────────────────
X_train = np.load(os.path.join(META_DIR, "liar_train_meta.npy"))
X_val   = np.load(os.path.join(META_DIR, "liar_val_meta.npy"))
X_test  = np.load(os.path.join(META_DIR, "liar_test_meta.npy"))

train_df = pd.read_csv(os.path.join(PROC_DIR, "liar_train.csv"))
val_df   = pd.read_csv(os.path.join(PROC_DIR, "liar_val.csv"))
test_df  = pd.read_csv(os.path.join(PROC_DIR, "liar_test.csv"))

y_train  = train_df["binary_label"].values.astype(np.int32)
y_val    = val_df  ["binary_label"].values.astype(np.int32)
y_test   = test_df ["binary_label"].values.astype(np.int32)

with open(os.path.join(META_DIR, "preprocessing_artifacts.pkl"), "rb") as f:
    artifacts = pickle.load(f)
FEAT_NAMES = artifacts["meta_features"]
CW         = artifacts["class_weights"]
SPW        = float(CW[0] / CW[1])

# ─────────────────────────────────────────────────────────────────────────────
# 2.  BASELINE XGBoost
# ─────────────────────────────────────────────────────────────────────────────
base_xgb = xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric="logloss",
                             use_label_encoder=False, scale_pos_weight=SPW, device="cpu")
base_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

base_test_pred = base_xgb.predict(X_test)
base_test_prob = base_xgb.predict_proba(X_test)
base_test_f1   = f1_score(y_test, base_test_pred, average="macro")
base_test_auc  = roc_auc_score(y_test, base_test_prob[:,1])

# ─────────────────────────────────────────────────────────────────────────────
# 3.  OPTUNA BAYESIAN HYPERPARAMETER SEARCH
# ─────────────────────────────────────────────────────────────────────────────
CV_FOLDS = 5
SKF      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

def objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators",     100, 1000, step=50),
        "max_depth"        : trial.suggest_int("max_depth",         3, 8),
        "learning_rate"    : trial.suggest_float("learning_rate",   1e-3, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample",       0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree",0.5, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight",  1, 10),
        "gamma"            : trial.suggest_float("gamma",           0.0, 5.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha",       1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda",      1e-4, 10.0, log=True),
        "scale_pos_weight" : SPW, "eval_metric": "logloss", "use_label_encoder": False,
        "random_state"     : 42, "device": "cpu",
    }
    clf = xgb.XGBClassifier(**params)
    scores = cross_val_score(clf, X_cv, y_cv, cv=SKF, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

print("="*70 + f"\n  OPTUNA SEARCH ({CV_FOLDS}-fold CV)\n" + "="*70)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=False)
best_params, best_cv_f1 = study.best_params, study.best_value

# ─────────────────────────────────────────────────────────────────────────────
# 4.  FINAL MODEL
# ─────────────────────────────────────────────────────────────────────────────
best_params.update({"scale_pos_weight": SPW, "eval_metric": "logloss",
                    "use_label_encoder": False, "random_state": 42, "device": "cpu", "tree_method": "hist"})

final_xgb = xgb.XGBClassifier(**best_params)
final_xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)

val_proba, test_proba  = final_xgb.predict_proba(X_val), final_xgb.predict_proba(X_test)
val_pred, test_pred   = val_proba.argmax(axis=1), test_proba.argmax(axis=1)
val_f1, test_f1     = f1_score(y_val, val_pred, average="macro"), f1_score(y_test, test_pred, average="macro")
val_auc, test_auc    = roc_auc_score(y_val, val_proba[:, 1]), roc_auc_score(y_test, test_proba[:, 1])
val_acc, test_acc    = accuracy_score(y_val, val_pred), accuracy_score(y_test, test_pred)

print(f"\n  Final Test — Acc: {test_acc:.4f}  F1: {test_f1:.4f}  AUC: {test_auc:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  EXTRACT PROBABILITY ARRAYS FOR STACKING
# ─────────────────────────────────────────────────────────────────────────────
xgb_proba_train = final_xgb.predict_proba(X_train)
for name, arr in [("train", xgb_proba_train), ("val", val_proba), ("test", test_proba)]:
    np.save(os.path.join(STACK_DIR, f"xgb_proba_{name}.npy"), arr)

# ─────────────────────────────────────────────────────────────────────────────
# 6.  SHAP ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
explainer    = shap.TreeExplainer(final_xgb)
shap_values  = explainer.shap_values(X_test)
base_value   = explainer.expected_value
shap_importance = pd.DataFrame({"feature": FEAT_NAMES, "mean_|shap|": np.abs(shap_values).mean(axis=0)}).sort_values("mean_|shap|", ascending=False).reset_index(drop=True)

np.save(os.path.join(XAI_DIR, "xgb_shap_values_test.npy"), shap_values)
shap_importance.to_csv(os.path.join(XAI_DIR, "xgb_shap_importance.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────────
# 7.  VISUALIZATIONS (Fixed for NumPy 2.0)
# ─────────────────────────────────────────────────────────────────────────────
optuna_df = study.trials_dataframe()
fig = plt.figure(figsize=(20, 16))
fig.suptitle("XGBoost Metadata Base Learner — Full Diagnostic", fontsize=15, fontweight="bold")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# SHAP Bar
ax0 = fig.add_subplot(gs[0, :2])
colors = ["#d62728" if "false" in f or "pants" in f or "extreme" in f else "#1f77b4" for f in shap_importance["feature"]]
ax0.barh(shap_importance["feature"][::-1], shap_importance["mean_|shap|"][::-1], color=colors[::-1], edgecolor="white")
ax0.set_title("Global Feature Importance (mean |SHAP value|)", fontsize=11, fontweight="bold")
ax0.axvline(0, c="k", lw=0.8)

# Confusion Matrix
ax1 = fig.add_subplot(gs[0, 2])
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt="d", cmap="Blues", xticklabels=["Fake","Real"], yticklabels=["Fake","Real"], ax=ax1)
ax1.set_title(f"Confusion Matrix (Test)\nF1={test_f1:.4f}  AUC={test_auc:.4f}")

# SHAP Beeswarm
ax2 = fig.add_subplot(gs[1, :])
top8_feat = shap_importance["feature"].head(8).tolist()
top8_idx  = [FEAT_NAMES.index(f) for f in top8_feat]
sv_top8, X_test_top8   = shap_values[:, top8_idx], X_test[:, top8_idx]

cmap = plt.cm.RdYlGn
for rank, (feat, col_idx) in enumerate(zip(top8_feat[::-1], top8_idx[::-1])):
    sv_col = sv_top8[:, len(top8_feat) - 1 - rank]
    fv_col = X_test_top8[:, len(top8_feat) - 1 - rank]
    # FIX: Replaced .ptp() with (max - min) for NumPy 2.0 compatibility
    normed = (fv_col - fv_col.min()) / ((fv_col.max() - fv_col.min()) + 1e-9)
    jitter = np.random.RandomState(rank).uniform(-0.2, 0.2, len(sv_col))
    ax2.scatter(sv_col, rank + jitter, c=cmap(normed), alpha=0.4, s=8, linewidths=0)
ax2.set_yticks(range(len(top8_feat))); ax2.set_yticklabels(top8_feat[::-1], fontsize=9)
ax2.axvline(0, c="k", lw=1)
ax2.set_title("SHAP Beeswarm (Test Set, top-8 features)", fontsize=11)
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0,1)), ax=ax2, label="Feature value (normalised)")

# ROC Curve
ax3 = fig.add_subplot(gs[2, 0])
fpr_x, tpr_x, _ = roc_curve(y_test, test_proba[:, 1])
ax3.plot(fpr_x, tpr_x, "b-", lw=2, label=f"XGBoost AUC={test_auc:.4f}")
try:
    bert_proba = np.load(os.path.join(STACK_DIR, "bert_proba_test.npy"))
    fpr_b, tpr_b, _ = roc_curve(y_test, bert_proba[:, 1])
    ax3.plot(fpr_b, tpr_b, "r-", lw=2, label=f"BERT AUC={roc_auc_score(y_test, bert_proba[:, 1]):.4f}")
except FileNotFoundError: pass
ax3.plot([0,1],[0,1],"k--",lw=1); ax3.set_title("ROC Curve: XGBoost vs BERT"); ax3.legend(fontsize=9)

# PR Curve
ax4 = fig.add_subplot(gs[2, 1])
pr_x, rc_x, _ = precision_recall_curve(y_test, test_proba[:, 1])
ax4.plot(rc_x, pr_x, "b-", lw=2, label="XGBoost")
try:
    pr_b, rc_b, _ = precision_recall_curve(y_test, bert_proba[:, 1])
    ax4.plot(rc_b, pr_b, "r-", lw=2, label="BERT")
except NameError: pass
ax4.axhline(y_test.mean(), ls="--", c="k", lw=1); ax4.set_title("Precision-Recall Curve"); ax4.legend(fontsize=9)

# Optuna History
ax5 = fig.add_subplot(gs[2, 2])
ax5.plot(optuna_df["number"], optuna_df["value"].values, "o", alpha=0.4, ms=4, c="#aec7e8", label="Trial F1")
ax5.plot(optuna_df["number"], optuna_df["value"].cummax().values, "-", lw=2.5, c="#1f77b4", label="Best so far")
ax5.set_title(f"Optuna Search (60 trials)"); ax5.legend(fontsize=9)

plt.savefig(os.path.join(VIS_DIR, "08_xgboost_diagnostic.png"), bbox_inches="tight")
plt.close(); print(f"\n  ✓ Fig 8 saved: 08_xgboost_diagnostic.png")

# ─────────────────────────────────────────────────────────────────────────────
# 8.  SAVE ALL ARTIFACTS
# ─────────────────────────────────────────────────────────────────────────────
final_xgb.save_model(os.path.join(MODEL_DIR, "xgb_best.json"))
with open(os.path.join(MODEL_DIR, "optuna_study.pkl"), "wb") as f: pickle.dump(study, f)
with open(os.path.join(RESULTS_DIR, "xgb_results.pkl"), "wb") as f:
    pickle.dump({"model": "XGBoost", "test_f1_macro": float(test_f1), "test_roc_auc": float(test_auc), "best_params": best_params}, f)

print(f"\n  ✓ Model saved, probabilities generated, Phase 2, Step 2 is complete.")

  OPTUNA SEARCH (5-fold CV)

  Final Test — Acc: 0.7277  F1: 0.7066  AUC: 0.8121

  ✓ Fig 8 saved: 08_xgboost_diagnostic.png

  ✓ Model saved, probabilities generated, Phase 2, Step 2 is complete.
